In [2]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegressionCV
from sklearn.inspection import permutation_importance

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest
from joblib import Memory
import tempfile
import shutil

Vamos a cargar primero el mapeo oficial de snomed a cie10 de la nomenclatura oficial de SNOMED

# Rutas

In [3]:
load_dotenv(find_dotenv())

PROJECT_ROOT = Path(find_dotenv()).parent.resolve()
print(f"PROJECT_ROOT set to: {PROJECT_ROOT}")

SNOMED_DIR = PROJECT_ROOT / 'despliegue_Docker' / 'snomed_intl'
DATA_DIR = PROJECT_ROOT / 'data'
print(f"SNOMED_DIR set to: {SNOMED_DIR}")
print(f"DATA_DIR set to: {DATA_DIR}")

PROJECT_ROOT set to: C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025
SNOMED_DIR set to: C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\despliegue_Docker\snomed_intl
DATA_DIR set to: C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\data


## Refset para cruce del diccionario SNOMED CT a CIE10

In [4]:
file_path = SNOMED_DIR / 'SnomedCT_InternationalRF2_PRODUCTION_20251001T120000Z' / 'Snapshot' / 'Refset' / 'Map' / 'der2_iisssccRefset_ExtendedMapSnapshot_INT_20251001.txt'


df_map = pd.read_csv(file_path, sep='\t', encoding='utf-8')
df_final = (
    df_map.loc[(df_map["active"] == 1) & (df_map["mapCategoryId"] == 447637006), ["referencedComponentId", "mapTarget"]]
    .rename(columns={"referencedComponentId": "concepto_SNOMED", "mapTarget": "conecpto_CIE"})
)

df_final.reset_index(drop=True, inplace=True)
df_final

,concepto_SNOMED,conecpto_CIE
0,254153009,Q79.8
1,16623961000119100,D61.1
2,246951001,H10.8
3,472759003,I27.2
4,53597009,L56.2
...,...,...
128509,1085421000119104,K56.6
128510,12246481000119106,C79.0
128511,1231624007,H49.0
128512,94152006,C78.8


Hemos extraido una tabla de referencias cruzadas de SNOMED CT a CIE-10 con solo códigos activos y mapeos únicos. Ahora gracias a este crossmap, podremos realizar el mapeo de nuestras variables a CIE-10

In [5]:
df_cod = pd.read_excel(DATA_DIR / 'codificaciones_identificadas.xlsx')

# Filtrar SNOMED_CT y extraer el código SNOMED quitando el prefijo C-
df_snomed_ct = df_cod.loc[df_cod["SISTEMA_CODIFICACION"].eq("SNOMED-CT")].copy()
df_snomed_ct["code"] = df_snomed_ct["code"].astype(str).str.replace(r"^C-", "", regex=True)

variables_snomed= df_snomed_ct["code"].unique()

print(f"Variables SNOMED-CT identificadas: {len(variables_snomed)}")

Variables SNOMED-CT identificadas: 425


Mapeamos las variables identificadas a CIE-10

In [6]:
# Mapear códigos SNOMED a CIE-10 usando el crossmap df_final
# Asegurar que ambas columnas sean del mismo tipo para el merge
df_snomed_ct["code"] = df_snomed_ct["code"].astype(str)
df_final["concepto_SNOMED"] = df_final["concepto_SNOMED"].astype(str)

df_snomed_cie10 = df_snomed_ct.merge(
    df_final,
    left_on="code",
    right_on="concepto_SNOMED",
    how="left"
)

# Renombrar columna para mayor claridad
df_snomed_cie10 = df_snomed_cie10.rename(columns={"conecpto_CIE": "CIE10_MAPEO"})

# Mostrar resultados
print(f"✅ Mapeo completado: {df_snomed_cie10.shape[0]} registros procesados")
print(f"   - Códigos SNOMED mapeados a CIE-10: {df_snomed_cie10['CIE10_MAPEO'].notna().sum()}")
print(f"   - Códigos sin mapeo: {df_snomed_cie10['CIE10_MAPEO'].isna().sum()}")
print(f"\n📊 Primeros registros de la tabla mapeada:\n")
display(df_snomed_cie10[["code", "SISTEMA_CODIFICACION", "CIE10_MAPEO"]].head(10))

df_snomed_cie10=df_snomed_cie10[["code", "name", "SISTEMA_CODIFICACION", "CIE10_MAPEO"]]

✅ Mapeo completado: 442 registros procesados
   - Códigos SNOMED mapeados a CIE-10: 192
   - Códigos sin mapeo: 250

📊 Primeros registros de la tabla mapeada:



,code,SISTEMA_CODIFICACION,CIE10_MAPEO
0,103579009,SNOMED-CT,NaN
1,125680007,SNOMED-CT,NaN
2,186034007,SNOMED-CT,NaN
3,263495000,SNOMED-CT,NaN
4,398070004,SNOMED-CT,NaN
5,424144002,SNOMED-CT,NaN
6,710824005,SNOMED-CT,NaN
7,73595000,SNOMED-CT,Z73.3
8,171207006,SNOMED-CT,NaN
9,454711000124102,SNOMED-CT,NaN


In [7]:
df_snomed_cie10.dropna(inplace=True)

In [8]:
df_snomed_cie10

,code,name,SISTEMA_CODIFICACION,CIE10_MAPEO
7,73595000,Stress (finding),SNOMED-CT,Z73.3
21,422650009,Social isolation (finding),SNOMED-CT,Z60.4
23,444814009,Viral sinusitis (disorder),SNOMED-CT,J01.9
24,706893006,Victim of intimate partner abuse (finding),SNOMED-CT,T74.9
26,424393004,Reports of violence in the environment (finding),SNOMED-CT,Z65.5
...,...,...,...,...
433,406602003,Infection caused by Staphylococcus aureus,SNOMED-CT,A49.0
434,427089005,Diabetes from Cystic Fibrosis,SNOMED-CT,E84.8
435,427089005,Diabetes from Cystic Fibrosis,SNOMED-CT,E13.9
437,707418001,Male infertility due to cystic fibrosis (disor...,SNOMED-CT,E84.8


In [9]:
print(f"Códigos CIE10 únicos mapeados: {df_snomed_cie10['CIE10_MAPEO'].nunique()}")

Códigos CIE10 únicos mapeados: 166


## Transformamos a JSON para agrupación de las nuevas variables

In [10]:
import json

# Creamos un diccionario vacío para almacenar el resultado final
resultado = {}

# Agrupamos el DataFrame por la columna del código CIE10
grupos_cie10 = df_snomed_cie10.groupby('CIE10_MAPEO')

for cie10_code, grupo in grupos_cie10:
    # Convertimos la columna 'code' a string para que la clave del JSON sea texto
    codigos_snomed = grupo['code'].astype(str)
    nombres_snomed = grupo['name']
    
    # Creamos el diccionario interno { "snomed_code": "snomed_name" }
    # zip() empareja los códigos con sus respectivos nombres
    diccionario_interno = dict(zip(codigos_snomed, nombres_snomed))
    
    # Asignamos este diccionario interno a la clave CIE10 principal
    resultado[str(cie10_code)] = diccionario_interno

# Convertimos el diccionario de Python a un string con formato JSON
# indent=4 le da el formato legible con sangrías
# ensure_ascii=False permite que se conserven correctamente los acentos o caracteres especiales
json_final = json.dumps(resultado, indent=4, ensure_ascii=False)

print(json_final)

{
    "A41.0": {
        "448417001": "Sepsis caused by Staphylococcus aureus"
    },
    "A41.5": {
        "448813005": "Sepsis caused by Pseudomonas (disorder)"
    },
    "A41.9": {
        "91302008": "Sepsis (disorder)"
    },
    "A49.0": {
        "406602003": "Infection caused by Staphylococcus aureus"
    },
    "A49.9": {
        "87628006": "Bacterial infectious disease (disorder)"
    },
    "B34.9": {
        "770349000": "Sepsis caused by virus (disorder)"
    },
    "B96.2": {
        "301011002": "Escherichia coli urinary tract infection"
    },
    "C18.8": {
        "109838007": "Overlapping malignant neoplasm of colon"
    },
    "C18.9": {
        "363406005": "Malignant tumor of colon",
        "93761005": "Primary malignant neoplasm of colon"
    },
    "C50.9": {
        "254837009": "Malignant neoplasm of breast (disorder)",
        "417181009": "Estrogen+Progesterone receptor Ag [Presence] in Tissue by Immune stain"
    },
    "C78.5": {
        "94260004": "S

In [11]:
diccionario = json.loads(json_final)

# 2. Medir la longitud (número de claves principales)
numero_claves = len(diccionario)

if numero_claves == df_snomed_cie10['CIE10_MAPEO'].nunique():
    print(f"El JSON tiene {numero_claves} claves de primer orden, coincidiendo con los códigos CIE10 únicos mapeados.")
else:
    print(f"El JSON tiene {numero_claves} claves de primer orden.")

El JSON tiene 166 claves de primer orden, coincidiendo con los códigos CIE10 únicos mapeados.


In [13]:
# Guardar el JSON a archivo
json_output_path = PROJECT_ROOT / 'Variables_codif' / 'SNOMED' / 'JSON' / 'feature_mapping_cie10_diagnosticos.json'

with open(json_output_path, 'w', encoding='utf-8') as f:
    f.write(json_final)

print(f"✅ JSON guardado en: {json_output_path}")
print(f"   - Grupos CIE-10 únicos: {len(diccionario)}")
print(f"   - Total de códigos SNOMED mapeados: {sum(len(v) for v in diccionario.values())}")

✅ JSON guardado en: C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\Variables_codif\SNOMED\JSON\feature_mapping_cie10_diagnosticos.json
   - Grupos CIE-10 únicos: 166
   - Total de códigos SNOMED mapeados: 192


In [16]:
import subprocess

script_path = PROJECT_ROOT / 'Variables_codif' / 'SNOMED' / 'Scripts' / 'json2csv_CIE10_diagnosticos.py'

print(f"Ejecutando script: {script_path}\n")
result = subprocess.run([str(PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'), str(script_path)], 
                       capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

# Verificar que el archivo se creó
output_csv = DATA_DIR / 'cie_diagnosticos.csv'
if output_csv.exists():
    import os
    file_size_mb = os.path.getsize(output_csv) / (1024 * 1024)
    print(f"\n✅ Archivo generado exitosamente: {output_csv}")
    print(f"   Tamaño: {file_size_mb:.2f} MB")
else:
    print(f"\n❌ Error: El archivo {output_csv} no se generó")

Ejecutando script: C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\Variables_codif\SNOMED\Scripts\json2csv_CIE10_diagnosticos.py

PROJECT_ROOT set to: C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025
Loading C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\data\synthea_30k.csv...
Loading SNOMED dictionary C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\data\codificaciones_identificadas.xlsx...
Found 167 distinct targeted SNOMED columns in dataset.

--- Summary of Data Types ---
Inferred_Data_Type
Pure Boolean (True only)    160
Empty (All Null)              6
Boolean (True/False)          1
Name: count, dtype: int64

Detailed report saved to C:\Users\rtrin\OneDrive\Escritorio\FORMACION\MASTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\Variables_codif\SNOMED\cie10_diagnosticos_data_types_report.xlsx

Applying data typing to the raw datase